# Creating your own benchmark and mechanism

This tutorial begins with the mathematical problem definition and only then
maps that model to the code abstraction. The goal is to create a new benchmark
(a multi-agent environment) and a new mechanism that both reuse the existing
mechanism algorithms and the bilevel optimization infrastructure.

Every code cell runs. The whole notebook executes in well under a minute and
does **not** train any policy: the point is to validate the benchmark equations
and the mechanism transforms *before* they are handed to RLlib and the outer
optimizer.

What you will build:

1. `RenewableResourceEnv`, a renewable common-pool resource benchmark on top of
   `MultiAgentRegulatedEnv`;
2. `ScarcityTaxMechanism`, a custom reward-channel mechanism that satisfies the
   full `Mechanism` optimizer contract;
3. a composition of the existing `QuotaMechanism` with the new tax;
4. analytical tests of the transition, of the mechanism, and of one complete
   regulated step against a real `World` actor.

**Reading order.** This is the second of the three tutorials. Read
`mechanism_algorithms.ipynb` first for the mechanism abstraction and the
reference fishery stack, then this notebook, then `visualization.ipynb` for
the metric schemas, queries and reporters that section 23 below wires into
the new benchmark.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Make the repository importable when the kernel is started from ``tutorials/``.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "core").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
from dataclasses import dataclass, field, replace
from typing import Any, Callable, Self

import numpy as np
from gymnasium import spaces

from core.envs.hooks import action, observation, reset, reward, transition
from core.envs.marl_regulated import MultiAgentRegulatedEnv
from core.mechanism.base import Mechanism
from core.types import MultiAgentDict
from core.utils import sigmoid

# 1. Start from the mathematical model

Before creating classes, define the multi-agent controlled dynamical system:

$$
\mathcal E
=
(
\mathcal S,
\{\mathcal A_i\}_{i=1}^{N},
P,
\{R_i\}_{i=1}^{N},
\{\mathcal O_i\}_{i=1}^{N},
\gamma
).
$$

You need to specify:

1. state $s_t\in\mathcal S$;
2. each agent action $a_{i,t}$;
3. transition $P(s_{t+1}\mid s_t,a_t)$;
4. intrinsic/base reward $R_i$;
5. observation function $O_i$;
6. initial-state/reset distribution;
7. horizon/termination semantics.

Only after the benchmark is mathematically clear should a mechanism be
introduced.

# 2. Separate benchmark dynamics from regulation

The benchmark answers:

> What happens in the world when delivered actions are executed?

The mechanism answers:

> What actions reach the world, what reward reaches the learner, and what
> information reaches the learner?

Mechanism intervention:

$$
o_t^* = \mathcal M_\theta^O(s_t,o_t),
\qquad
a_t^* = \mathcal M_\theta^A(s_t,a_t),
\qquad
r_t^* = \mathcal M_\theta^R(r_t, s_t, a_t^*, s_{t+1}).
$$

The transition must consume the regulated action:

$$
s_{t+1} \sim P(\cdot\mid s_t,a_t^*).
$$

Keeping this boundary clean is what makes one quota algorithm reusable in a
fishery, irrigation system, energy market, traffic network, or another
benchmark.

`MultiAgentRegulatedEnv.step` implements exactly this order:

```text
policy output
    -> normalize action               (sigmoid squashing, see section 19)
    -> benchmark @action hook          (optional)
    -> mechanism.action                M^A
    -> benchmark @reward hook          (intrinsic / base reward on s_t, a_t*)
    -> benchmark @transition hook      s_{t+1} = P(s_t, a_t*)
    -> mechanism.reward                M^R  (bindings now see s_{t+1})
    -> benchmark @observation hook     o_i = O_i(s_{t+1})
    -> append mechanism.to_vector()    theta visible to agents
    -> mechanism.observation           M^O
    -> publish EnvStepContext          to the World actor
```

One consequence matters for reward-shaping mechanisms: their bindings are
resolved **after** the transition, so a "resource level" read in
`mechanism.reward` is the post-transition level $s_{t+1}$, consistent with
$r_t^* = \mathcal M_\theta^R(r_t, s_t, a_t^*, s_{t+1})$. Section 18 checks this
numerically.

# 3. Ingredients of a custom benchmark

A benchmark needs:

```text
1. state representation                  (self.S_t, a plain dict)
2. agent IDs / multi-agent mappings      (agents=[...])
3. action semantics                      (documented component map)
4. reset hook                            (@reset)
5. transition hook                       (@transition)
6. intrinsic/base reward hook            (@reward)
7. observation hook                      (@observation)
8. action and observation spaces         (gymnasium Box per agent)
9. runtime context / info values         (self._update_infos)
10. mechanism bindings                   (env -> value callables)
```

A custom mechanism needs:

```text
1. semantic parameters                   (dataclass fields)
2. dimension                             (property)
3. encode()
4. decode(x)
5. clip()
6. param_names()
7. to_vector()
8. optional action()
9. optional reward()
10. optional observation()
11. runtime bindings when benchmark context is required
```

# 4. Hooks

The decorators in `core.envs.hooks` mark benchmark-specific methods. Their
signatures are fixed by the base class:

```python
@reset
def initialize_state(self) -> dict:                       # -> S_0
    ...

@action                                                    # optional
def benchmark_action_preprocessing(self, action_dict) -> action_dict:
    ...

@reward
def intrinsic_reward(self, action_dict) -> reward_dict:    # on S_t and a_t*
    ...

@transition
def dynamics(self, *, A_t, S_t, **kwargs) -> dict:         # -> S_{t+1}
    ...

@observation
def make_observation(self, observation_dict) -> observation_dict:  # o_i = O_i(S_t)
    ...
```

All five decorators are imported in the import cell at the top; the worked
benchmark below declares no `@action` hook, so `action` is imported for the
example only. `MultiAgentRegulatedEnv.__init_subclass__` discovers the decorated
methods.
A class may declare **exactly one** method per hook type; declaring two raises
`TypeError` at class-definition time rather than silently overriding one.

That lets the generic environment own the lifecycle while a benchmark owns
only domain-specific equations.

In [ ]:
# A second @reset hook is rejected when the class body is executed.
try:

    class TwoResets(MultiAgentRegulatedEnv):
        @reset
        def first(self):
            return {}

        @reset
        def second(self):
            return {}

except TypeError as err:
    print(f"TypeError as expected: {err}")

# 5. Runtime context and diagnostics

Each regulated step publishes an `EnvStepContext` to the shared `World` actor
with the values that actually drove the step:

```text
env_id, seed, policy_seed, status, mechanism (candidate index),
observation, observation_map, reward, action, info
```

Domain-specific diagnostics belong in `info`, recorded from inside the hooks
with `self._update_infos(key, values)` (a scalar is broadcast to every agent;
a per-agent dict is stored as is), for example:

```text
resource stock, biological growth, realized harvest, restoration,
quota violation, reservoir level
```

The runtime context answers "what happened on this step?". How a quantity is
accumulated and reduced for reporting is a separate concern handled by the
reporting layer; keep the two responsibilities apart and do not compute
summary statistics inside the hooks.

# 6. Worked benchmark: renewable common-pool resource

We create a minimal renewable resource.

State:

$$
x_t\in[0,K].
$$

Normalized state:

$$
\bar x_t=\frac{x_t}{K}.
$$

Each agent chooses one extraction fraction:

$$
a_{i,t} = \begin{bmatrix} u_{i,t} \end{bmatrix},
\qquad
u_{i,t}\in[0,1].
$$

Logistic growth:

$$
G(x_t) = r x_t \left( 1-\frac{x_t}{K} \right).
$$

Maximum extraction capacity for agent $i$ is $H_i^{\max}$; attempted extraction:

$$
H_{i,t}=u_{i,t}H_i^{\max}.
$$

Transition:

$$
x_{t+1}
=
\operatorname{clip}
\left(
x_t+G(x_t)-\sum_iH_{i,t},
0,
K
\right).
$$

Base reward, the delivered extraction normalized by the agent's capacity:

$$
r_{i,t}=\frac{H_{i,t}}{H_i^{\max}}=u_{i,t}.
$$

Normalizing keeps the intrinsic reward in $[0,1]$, on the same scale as the
mechanism terms defined later (this is also what the fishery benchmark does).

Observation:

$$
o_{i,t}=[\bar x_t].
$$

# 7. Implement the benchmark class

Keep the constructor benchmark-specific: it parses the ecology configuration
and nothing else. Do not encode quota/subsidy logic directly inside it.

The base-class constructor takes `world, mechanism_id, agents` and, optionally,
`mechanism` (a template used until a candidate is published), `horizon`, `seed`,
`policy_seed`, `mode`, `action_temperature`, `action_spaces`,
`observation_spaces`. Benchmark-specific keyword arguments (here
`ecology_cfg`) are consumed by the subclass; everything else is forwarded with
`**kwargs`.

The four hooks are shown one at a time below and assembled into a single class
in section 7.5.

In [ ]:
# Document the action layout once, and reference it everywhere (section 8).
EXTRACTION_COMPONENT = 0

## 7.1 Reset hook

Mathematically:

$$
x_0\sim p_0(x).
$$

Begin with a deterministic reset so the model can be validated exactly. A
stochastic reset would draw from `self.rng`, the generator seeded with the
constructor's `seed`.

```python
@reset
def reset_resource(self) -> dict[str, float]:
    return {"resource": self.x_init, "last_usage": 0.0}
```

## 7.2 Transition hook

The transition receives the **already regulated action** $a_t^*$ as `A_t`
and the current state as `S_t` (a copy), and returns $s_{t+1}$. It is also the
natural place to record diagnostics with `self._update_infos`.

That is the core mechanism-design contract.

```python
@transition
def renewable_transition(self, *, A_t, S_t, **kwargs) -> dict[str, float]:
    x = float(S_t["resource"])
    requested = {aid: self._extraction_fraction(a) * self.max_harvest_per_agent
                 for aid, a in A_t.items()}
    H = float(sum(requested.values()))
    growth = self.r * x * (1.0 - x / self.K)
    available = max(x + growth, 0.0)
    realized = min(H, available)
    x_next = float(np.clip(available - realized, 0.0, self.K))
    ...
    return {"resource": x_next, "last_usage": realized}
```

## 7.3 Base reward hook

The benchmark defines the intrinsic reward **before reward shaping**. The hook
receives the delivered actions $a_t^*$ and is evaluated on the current state
$s_t$, before the transition:

$$
r_{i,t}=u_{i,t}.
$$

```python
@reward
def harvest_reward(self, action_dict) -> MultiAgentDict:
    return {aid: self._extraction_fraction(a) for aid, a in action_dict.items()}
```

## 7.4 Observation hook

The benchmark observation is intentionally minimal:

$$
o_{i,t}=[x_t/K].
$$

The base class then appends `mechanism.to_vector()` and applies
`mechanism.observation`, so mechanisms can augment it later.

```python
@observation
def resource_observation(self, observation_dict) -> MultiAgentDict:
    obs = np.array([self.S_t["resource"] / self.K], dtype=np.float32)
    return {aid: obs.copy() for aid in self.agents}
```

## 7.5 The complete class

In [ ]:
class RenewableResourceEnv(MultiAgentRegulatedEnv):
    """Logistic common-pool resource shared by ``N`` extractors.

    Parameters
    ----------
    ecology_cfg : dict
        ``r`` (growth rate, default 0.3), ``K`` (carrying capacity, default
        1000), ``x_init`` (initial stock, default ``0.8 K``) and
        ``max_harvest_per_agent`` (``H_i^max``, default ``0.02 K``).
    **kwargs
        Forwarded to ``MultiAgentRegulatedEnv`` (world, mechanism_id, agents,
        mechanism, horizon, seed, policy_seed, spaces, ...).
    """

    def __init__(self, *, ecology_cfg: dict[str, Any], **kwargs):
        super().__init__(**kwargs)

        self.r = float(ecology_cfg.get("r", 0.3))
        self.K = float(ecology_cfg.get("K", 1_000.0))
        self.x_init = float(ecology_cfg.get("x_init", 0.8 * self.K))
        self.max_harvest_per_agent = float(
            ecology_cfg.get("max_harvest_per_agent", 0.02 * self.K)
        )

        # Names of the benchmark observation components (published in contexts).
        self.obs_map = ["resource_norm"]

    # --- helpers -------------------------------------------------------------

    @staticmethod
    def _extraction_fraction(action) -> float:
        """Read ``u_i`` from a delivered action, using the documented layout."""
        a = np.asarray(action, dtype=np.float32).reshape(-1)
        return float(a[EXTRACTION_COMPONENT])

    # --- hooks ---------------------------------------------------------------

    @reset
    def reset_resource(self) -> dict[str, float]:
        return {"resource": self.x_init, "last_usage": 0.0}

    @reward
    def harvest_reward(self, action_dict: MultiAgentDict) -> MultiAgentDict:
        """Intrinsic reward ``r_i = u_i`` (extraction normalized by capacity)."""
        return {
            agent_id: self._extraction_fraction(action)
            for agent_id, action in action_dict.items()
        }

    @transition
    def renewable_transition(
        self, *, A_t: MultiAgentDict, S_t: dict[str, float], **kwargs
    ) -> dict[str, float]:
        x = float(S_t["resource"])

        requested = {
            agent_id: self._extraction_fraction(action) * self.max_harvest_per_agent
            for agent_id, action in A_t.items()
        }
        H = float(sum(requested.values()))

        growth = self.r * x * (1.0 - x / self.K)
        available = max(x + growth, 0.0)
        realized = min(H, available)
        x_next = float(np.clip(available - realized, 0.0, self.K))

        self._update_infos(key="resource", values=x)
        self._update_infos(key="resource_norm", values=x / self.K)
        self._update_infos(key="growth", values=growth)
        self._update_infos(key="realized_harvest", values=realized)

        return {"resource": x_next, "last_usage": realized}

    @observation
    def resource_observation(self, observation_dict: MultiAgentDict) -> MultiAgentDict:
        obs = np.array([self.S_t["resource"] / self.K], dtype=np.float32)
        return {agent_id: obs.copy() for agent_id in self.agents}


print(
    "hooks discovered:",
    {
        hook: getattr(RenewableResourceEnv, f"_{hook}")
        for hook in ("reset", "action", "reward", "transition", "observation")
    },
)

# 8. Define action semantics explicitly

Document every action component.

For the example:

```text
component 0 -> extraction fraction        (EXTRACTION_COMPONENT)
```

If the benchmark later becomes:

```text
component 0 -> extraction fraction
component 1 -> restoration effort
component 2 -> investment
```

then mechanism `action_component` values must reference this map.

Never make a mechanism infer action meaning from array position without a
documented contract.

# 9. Add an existing quota mechanism

The generic `QuotaMechanism` asks only for a normalized resource level:

$$
b_t=x_t/K.
$$

It is supplied through a **binding**: a callable `env -> value` that the
environment resolves at every step and passes to the mechanism's channel
methods as a keyword argument.

In [ ]:
from core.mechanism.algorithms.quota import QuotaMechanism


def resource_level(env: RenewableResourceEnv) -> float:
    """Binding: normalized stock ``x_t / K`` read from the benchmark state."""
    return env.S_t["resource"] / env.K


quota = QuotaMechanism(
    fixed_quota=0.50,
    action_component=EXTRACTION_COMPONENT,
    bindings={"resource_level": resource_level},
)

print("dimension:", quota.dimension, "| params:", quota.param_names())
print("encode():", quota.encode(), "| to_vector():", quota.to_vector())

This is the abstraction boundary:

```text
Quota:
    "I require a normalized resource_level."

Benchmark:
    "For me, resource_level = resource / K."
```

The quota therefore does not need to know whether the resource is fish,
groundwater, a reservoir, or another stock.

# 10. Create a custom mechanism from mathematics

We create a scarcity-weighted extraction tax acting on the reward channel.

## 10.1 Mathematical definition

Let $\lambda\in[0,1]$ be the tax rate and $\bar x_{t+1}=x_{t+1}/K$ the
normalized stock **after** the transition (the reward channel runs after the
dynamics, see section 2). For delivered extraction fraction $u_{i,t}$:

$$
r_{i,t}^{*}
=
r_{i,t}
-
\lambda\,(1-\bar x_{t+1})\,u_{i,t}.
$$

Properties:

- high resource -> small tax;
- low resource -> larger tax;
- zero extraction -> zero tax;
- the mechanism does not change the physical dynamics.

## 10.2 Implement the custom mechanism

A custom mechanism should satisfy the complete `Mechanism` optimizer contract.
House style, followed by `ThresholdPenaltyMechanism` (fixed, dimension 0) and
`QuotaMechanism` (optimized, dimension 1) in `core/mechanism/algorithms`:

- a `@dataclass(frozen=True)` with a `bindings` field excluded from `repr` and
  comparison;
- validation in `__post_init__` with `ValueError` (never `assert`);
- `decode` calls `self._validate(x)` (shape and finiteness) and returns a new
  instance through `dataclasses.replace`;
- channel methods accept `**kwargs` and read what they need
  (`resource_level` from the binding, `action_after` supplied by the
  environment on the reward channel).

In [ ]:
@dataclass(frozen=True)
class ScarcityTaxMechanism(Mechanism):
    """Scarcity-weighted extraction tax on the reward channel.

    ``r*_i = r_i - tax_rate * (1 - resource_level) * u_i`` where ``u_i`` is the
    delivered extraction fraction on ``action_component``.

    Parameters
    ----------
    tax_rate : float
        Tax rate in ``[0, 1]``; the single optimized parameter.
    action_component : int
        Index of the extraction fraction in the delivered action.
    bindings : dict
        Must provide ``"resource_level"``: ``env -> float`` in ``[0, 1]``.
    """

    tax_rate: float
    action_component: int = EXTRACTION_COMPONENT

    bindings: dict[str, Callable[[Any], Any]] = field(
        default_factory=dict, repr=False, compare=False
    )

    def __post_init__(self) -> None:
        if "resource_level" not in self.bindings:
            raise ValueError(
                "ScarcityTaxMechanism requires a 'resource_level' binding."
            )
        if not 0.0 <= self.tax_rate <= 1.0:
            raise ValueError(f"tax_rate must be in [0, 1], got {self.tax_rate}")

    # --- optimizer-space API -------------------------------------------------

    @property
    def dimension(self) -> int:
        return 1

    def encode(self) -> np.ndarray:
        return np.array([self.tax_rate], dtype=np.float32)

    def decode(self, x: np.ndarray) -> Self:
        x = self._validate(x)
        return replace(self, tax_rate=float(x[0]))

    def clip(self) -> Self:
        return replace(self, tax_rate=float(np.clip(self.tax_rate, 0.0, 1.0)))

    def param_names(self) -> list[str]:
        return ["tax_rate"]

    def to_vector(self) -> np.ndarray:
        return np.array([self.tax_rate], dtype=np.float32)

    # --- channels -------------------------------------------------------------

    def tax(self, resource_level: float, extraction_fraction: float) -> float:
        """Reward deduction for one agent."""
        scarcity = 1.0 - float(resource_level)
        return float(self.tax_rate * scarcity * extraction_fraction)

    def reward(self, reward_dict: MultiAgentDict, **kwargs) -> MultiAgentDict:
        resource_level = kwargs["resource_level"]  # from the binding
        actions = kwargs["action_after"]  # delivered actions, from the env
        return {
            agent_id: float(
                r
                - self.tax(
                    resource_level, float(actions[agent_id][self.action_component])
                )
            )
            for agent_id, r in reward_dict.items()
        }


tax = ScarcityTaxMechanism(tax_rate=0.5, bindings={"resource_level": resource_level})
tax

In [ ]:
# Validation happens at construction and raises ValueError, never assert.
for bad_kwargs in (
    {"tax_rate": 1.5, "bindings": {"resource_level": resource_level}},
    {"tax_rate": 0.5},
):
    try:
        ScarcityTaxMechanism(**bad_kwargs)
    except ValueError as err:
        print(f"ValueError: {err}")

# 11. Why each mechanism method exists

`dimension`
: Number of optimizer-controlled scalar dimensions.

`encode()`
: Mechanism -> optimizer vector in $[0,1]^d$.

`decode(x)`
: Optimizer vector -> new immutable mechanism instance. The outer optimizer
  calls `template.decode(x)` on every candidate it samples.

`clip()`
: Return a copy whose parameters lie within their valid semantic bounds.

`param_names()`
: Names that correspond exactly to the entries of `encode()`.

`to_vector()`
: Semantic mechanism representation appended to every agent observation. It
  may include fixed (non-optimized) parameters.

`encode()` and `to_vector()` happen to match for a simple mechanism, but they
serve different purposes: the first is the optimizer's view, the second is the
agents' view.

# 12. Fixed versus optimized mechanisms

A fixed mechanism exposes $d=0$:

```python
dimension == 0
encode() -> np.empty(0, dtype=np.float32)
decode(np.empty(0)) -> self
param_names() -> []
```

`ThresholdPenaltyMechanism` is the reference implementation of this pattern.

An optimized mechanism has $d>0$ (`QuotaMechanism`, `ScarcityTaxMechanism`).

Composition adds dimensions:

$$
d_{\text{composition}} = \sum_i d_i .
$$

# 13. Compose the quota and custom tax

`ChainedMechanism` applies its children in order on every channel; each child
resolves its **own** bindings against the environment. Its optimizer vector is
the concatenation of the children's vectors, sliced back on `decode`.

Action path:

```text
policy action -> quota -> tax (identity action) -> transition
```

Reward path:

```text
base reward -> quota (identity reward) -> scarcity tax -> learner
```

Because these two mechanisms alter different channels, their order has no
effect here. If two mechanisms both transform reward or action, chain order can
change the result.

`ParallelMechanism(children, action_merge, reward_merge, observation_merge)` is
the alternative: every child receives the same input and one merge function
per channel combines the outputs.

In [ ]:
from core.mechanism.composition.chained_mechanism import ChainedMechanism

mechanism = ChainedMechanism(children=(quota, tax))

print("dimension :", mechanism.dimension)
print("params    :", mechanism.param_names())
print("encode()  :", mechanism.encode())
print("to_vector():", mechanism.to_vector())

assert mechanism.dimension == quota.dimension + tax.dimension

# 14. Bindings are dependency injection

Avoid benchmark-specific code inside a generic mechanism:

```python
env.S_t["fish"]          # inside a mechanism: wrong layer
```

Prefer a semantic dependency:

```python
bindings={"resource_level": lambda env: ...}
```

A mechanism can request:

```text
resource_level, previous_actions, agent_ids, demand, capacity, price,
network_load, ...
```

The benchmark decides how those values are computed. `Mechanism.resolve(env)`
evaluates every binding and the environment passes the result as keyword
arguments to `action`, `reward` and `observation`. The environment adds
`env=self` on every channel and `action_after=<delivered actions>` on the
reward channel.

# 15. Test the benchmark equation before RL

Do not start with APPO or ES.

First test the transition as a pure mathematical function, then check that the
hook reproduces it. The hook can be called directly on an environment instance
without any Ray runtime: construction only stores the `world` handle, so
`world=None` is fine as long as `reset`/`step` are not called.

In [ ]:
def logistic_step(x: float, harvest: float, *, r: float, K: float) -> float:
    """Reference implementation of ``x_{t+1} = clip(x + r x (1 - x/K) - H, 0, K)``."""
    growth = r * x * (1.0 - x / K)
    return float(np.clip(x + growth - harvest, 0.0, K))


x_next = logistic_step(800.0, 50.0, r=0.3, K=1_000.0)
x_next

Manual verification:

$$
G(800) = 0.3\cdot 800\cdot(1-0.8) = 48,
\qquad
x_{t+1} = 800+48-50 = 798.
$$

In [ ]:
assert np.isclose(x_next, 798.0)

# The same equation through the hook, with no World and no Ray.
AGENTS = ["u0", "u1"]
ECOLOGY = {"r": 0.3, "K": 1_000.0, "x_init": 800.0, "max_harvest_per_agent": 50.0}

bare_env = RenewableResourceEnv(
    world=None, mechanism_id=0, agents=AGENTS, ecology_cfg=ECOLOGY
)
S_next = bare_env.renewable_transition(
    A_t={
        "u0": np.array([1.0], dtype=np.float32),
        "u1": np.array([0.0], dtype=np.float32),
    },
    S_t={"resource": 800.0, "last_usage": 0.0},
)
print(S_next)
assert np.isclose(S_next["resource"], 798.0)
assert np.isclose(S_next["last_usage"], 50.0)

# Boundary: extraction cannot exceed what is available.
S_empty = bare_env.renewable_transition(
    A_t={
        "u0": np.array([1.0], dtype=np.float32),
        "u1": np.array([1.0], dtype=np.float32),
    },
    S_t={"resource": 10.0, "last_usage": 0.0},
)
assert S_empty["resource"] == 0.0
assert np.isclose(S_empty["last_usage"], 10.0 + 0.3 * 10.0 * 0.99)

# 16. Test the custom mechanism analytically

Call the reward channel directly with the keyword arguments the environment
would supply. Choose:

```text
tax_rate       = 0.5
resource_level = 0.2   (scarcity = 0.8)
action         = 0.4
base reward    = 1.0
```

Expected:

$$
r^* = 1.0 - 0.5\cdot 0.8\cdot 0.4 = 0.84 .
$$

In [ ]:
shaped = tax.reward(
    {"u0": 1.0},
    resource_level=0.2,
    action_after={"u0": np.array([0.4], dtype=np.float32)},
)
print(shaped)
assert np.isclose(shaped["u0"], 0.84)

# Zero extraction -> zero tax, whatever the scarcity.
untaxed = tax.reward(
    {"u0": 1.0}, resource_level=0.0, action_after={"u0": np.zeros(1, dtype=np.float32)}
)
assert untaxed["u0"] == 1.0

Every mechanism should have an analytical test like this before it is used in
a distributed RL run.

# 17. Test encode/decode round trips

For every optimized mechanism:

$$
M \xrightarrow{\text{encode}} x \xrightarrow{\text{decode}} M'
$$

should preserve semantic parameters up to floating-point tolerance, and the
composed mechanism must slice the vector back to the right child.

In [ ]:
x = mechanism.encode()
reconstructed = mechanism.decode(x)

np.testing.assert_allclose(reconstructed.encode(), x)
assert reconstructed.param_names() == mechanism.param_names()

# A new point in optimizer space lands on the right child parameters.
candidate = mechanism.decode(np.array([0.3, 0.9], dtype=np.float32))
assert np.isclose(candidate.children[0].fixed_quota, 0.3)
assert np.isclose(candidate.children[1].tax_rate, 0.9)

# Wrong shape is a ValueError from Mechanism._validate.
try:
    tax.decode(np.array([0.1, 0.2], dtype=np.float32))
except ValueError as err:
    print(f"ValueError: {err}")

# clip() returns a copy inside the semantic bounds.
assert tax.clip() == tax

# 18. Test one complete regulated step

`reset` and `step` talk to the `World` actor: the environment fetches the
mechanism published for its `mechanism_id` (and matching `policy_seed`) at
`reset`, and publishes an `EnvStepContext` at every `step`. In a bilevel run the
outer optimizer publishes candidates; here we publish one by hand, as
`tests/integration/test_fishery_mechanisms.py::test_stage_a_environment_only`
does.

`RayRuntime.ensure_initialized` starts Ray the way the framework does (it also
works around Ray's handling of drivers launched with `uv run`).

In [ ]:
import ray

from core.adaptors.ray.runtime import RayRuntime, RayRuntimeConfig
from core.world.base import World
from core.world.context import (
    Context,
    EnvStepContext,
    MechanismContext,
    MechanismStatus,
)

RayRuntime.ensure_initialized(RayRuntimeConfig(num_cpus=2))
world = World.options(name="tutorial_world_benchmark").remote()

POLICY_SEED = 123

published = Context(
    id=None,
    opt_id="tutorial_outer",
    step=0,
    env="tutorial",
    payload=MechanismContext(
        index=0,  # this is the mechanism_id the environment will ask for
        env_id=None,
        seed=POLICY_SEED,
        status=MechanismStatus.published,
        mechanism=mechanism,
        metrics=None,
    ),
)
published_id = ray.get(world.append_context.remote(published))
print("published context id:", published_id)

Deterministic fixture:

1. start from $x_0 = 800$, $K = 1000$, $r = 0.3$, $H^{\max}_i = 20$, two agents;
2. supply the raw policy output $z = 0$ for both agents;
3. expected normalized action $u = \sigma(0/4) = 0.5$;
4. expected quota transform: at $b = 0.8$ with `fixed_quota = 0.5` the allowed
   fraction is $\approx 1$, so the request $0.5$ passes unchanged;
5. expected transition: $x_1 = 800 + 48 - 2\cdot 0.5\cdot 20 = 828$;
6. expected base reward: $r_i = u = 0.5$;
7. expected shaped reward: $r_i^* = 0.5 - 0.5\,(1-0.828)\,0.5 = 0.457$;
8. expected observation: $[x_1/K,\ \text{fixed\_quota},\ \text{tax\_rate},\ \text{allowed\_frac}]$.

Each intermediate value is compared. This is much easier to debug than checking
only a final episode return.

In [ ]:
FIXTURE = {"r": 0.3, "K": 1_000.0, "x_init": 800.0, "max_harvest_per_agent": 20.0}

# benchmark obs (1) + mechanism.to_vector() (2) + quota's allowed_frac (1)
OBS_DIM = 1 + mechanism.to_vector().shape[0] + 1

env = RenewableResourceEnv(
    world=world,
    opt_id="tutorial_inner",
    mechanism_id=0,
    agents=AGENTS,
    mechanism=mechanism,
    horizon=5,
    seed=7,
    policy_seed=POLICY_SEED,
    ecology_cfg=FIXTURE,
    action_spaces={a: spaces.Box(-np.inf, np.inf, (1,), np.float32) for a in AGENTS},
    observation_spaces={
        a: spaces.Box(0.0, 1.0, (OBS_DIM,), np.float32) for a in AGENTS
    },
)

obs, infos = env.reset()
assert env.published_mechanism_assigned, "the World did not hand back the candidate"
print("reset obs:", obs)

raw_action = {a: np.zeros(1, dtype=np.float32) for a in AGENTS}

# 3. normalization
u = sigmoid(0.0 / env.action_temperature)
assert np.isclose(u, 0.5)

# 4. quota transform at b = 0.8
allowed = quota.allowed_fraction(0.8)
print("allowed fraction at b=0.8:", allowed)
assert allowed > 0.999 and u < allowed

obs, rewards, terminated, truncated, infos = env.step(raw_action)

# 5. transition
x_1 = logistic_step(800.0, 2 * u * FIXTURE["max_harvest_per_agent"], r=0.3, K=1_000.0)
assert np.isclose(x_1, 828.0)
assert np.isclose(env.S_t["resource"], x_1)

# 6. base reward (recorded by the base class in infos)
assert all(np.isclose(infos[a]["intrinsic_utility"], u) for a in AGENTS)

# 7. shaped reward: the tax reads the POST-transition stock x_1 / K
expected_shaped = u - 0.5 * (1.0 - x_1 / 1_000.0) * u
print("shaped rewards:", rewards, "| expected:", expected_shaped)
assert all(np.isclose(rewards[a], expected_shaped, atol=1e-6) for a in AGENTS)

# 8. observation layout
for a in AGENTS:
    np.testing.assert_allclose(
        obs[a], [x_1 / 1_000.0, 0.5, 0.5, allowed], rtol=1e-5, atol=1e-6
    )
assert not terminated["__all__"] and not truncated["__all__"]

# 19. Action normalization

The regulated environment maps raw policy outputs $z\in\mathbb R$ through:

$$
a=\sigma(z/T),
$$

with $T$ the constructor argument `action_temperature` (default $4$).

Mechanisms therefore receive normalized semantic action components in
$[0,1]$. If your benchmark needs another physical range, add an explicit
conversion in the benchmark (as `max_harvest_per_agent` does above). Do not
make mechanisms guess whether the input is a logit, a normalized fraction, or a
physical unit.

In [ ]:
for z in (-8.0, 0.0, 8.0):
    print(f"z={z:+.1f} -> u={sigmoid(z / env.action_temperature):.4f}")

# 20. Observation spaces must include mechanism augmentation

If $o_t\in\mathbb R^{d_o}$ and the mechanism layer appends $d_m$ features:

$$
o_t^*\in\mathbb R^{d_o+d_m}.
$$

Here $d_m$ = `len(mechanism.to_vector())` plus whatever `mechanism.observation`
appends (`QuotaMechanism` adds its allowed fraction; the social-influence
mechanism adds $(N-1)d_a$ peer actions). The declared Gymnasium space must
match, otherwise RLlib's environment check fails before training.

In [ ]:
obs, _ = env.reset()
for agent_id, value in obs.items():
    assert value.shape == (OBS_DIM,)
    assert env.observation_spaces[agent_id].contains(value), (agent_id, value)
print("observation dimension:", OBS_DIM, "| spaces OK")

# 21. Context publication

A regulated step publishes the values actually used by the system. Run a full
seeded episode and read the contexts back from the World: one `EnvStepContext`
per step, tagged with the environment's `opt_id`.

If the published context disagreed with the values that drove the transition or
the learner, debugging and reproducibility would become unreliable.

In [ ]:
before = len(ray.get(world.get_opt_ctx_ids.remote("tutorial_inner")))

obs, _ = env.reset()
policy_rng = np.random.default_rng(0)
for step in range(env.horizon):
    obs, rewards, terminated, truncated, infos = env.step(
        {a: policy_rng.normal(size=1).astype(np.float32) for a in AGENTS}
    )
    assert all(np.isfinite(r) for r in rewards.values())
    assert truncated["__all__"] is (step == env.horizon - 1)

ctx_ids = ray.get(world.get_opt_ctx_ids.remote("tutorial_inner"))
assert len(ctx_ids) - before == env.horizon

last = ray.get(world.get_context.remote(ctx_ids[-1])).payload
assert isinstance(last, EnvStepContext)
assert last.mechanism == 0 and last.policy_seed == POLICY_SEED and last.seed == 7
print("observation_map:", last.observation_map)
print("info keys     :", sorted(last.info["u0"]))
print("final stock   :", env.S_t)

# 22. The inert step before publication

Until a mechanism is published for its `mechanism_id`, the environment falls
back to the `mechanism` template for observation sizes and `step` is inert:
zero rewards, no dynamics, no context publication. RLlib runs its environment
checks (reset, a few random steps, space validation) before any candidate
exists, and this fallback is what lets those checks pass.

In [ ]:
unpublished = RenewableResourceEnv(
    world=world,
    mechanism_id=99,  # nothing published under this id
    agents=AGENTS,
    mechanism=mechanism,
    horizon=5,
    seed=7,
    policy_seed=POLICY_SEED,
    ecology_cfg=FIXTURE,
    action_spaces={a: spaces.Box(-np.inf, np.inf, (1,), np.float32) for a in AGENTS},
    observation_spaces={
        a: spaces.Box(0.0, 1.0, (OBS_DIM,), np.float32) for a in AGENTS
    },
)
obs, _ = unpublished.reset()
assert not unpublished.published_mechanism_assigned
assert all(o.shape == (OBS_DIM,) for o in obs.values())

obs, rewards, *_ = unpublished.step(raw_action)
assert rewards == {a: 0.0 for a in AGENTS}
assert unpublished.S_t["resource"] == FIXTURE["x_init"]  # no dynamics applied
print("inert step OK:", rewards)

# 23. Log benchmark metrics into a schema

The environments above publish contexts but log no metric: `RenewableResourceEnv`
was built without a `schema`, so `env.logger` is `None` and `self._log(...)` is
a no-op. The metrics stack (the subject of `visualization.ipynb`) needs three
things from a benchmark, all optional and all forwarded by `**kwargs` to
`MultiAgentRegulatedEnv`:

- `schema=`: a `MetricSchema` subclass; the environment then owns
  `env.logger = MetricLogger.from_schema(schema)`. Subclass
  `EpisodeRolloutSchema` so that the fields the base class already pushes
  (`iter`, `env_id`, `mechanism_id`, `seed`, `policy_seed`, `reward_mean` and
  `by_agent/<id>/reward`) have a home;
- `queries=`: the `Query` tuple an environment-level reporter renders at the
  end of each episode (`core/callbacks.py::log_and_report_episode_metrics`);
- `reporter_cfg=`: the backend (`CSVConfig`, `WandbConfig`); in a bilevel run it
  is injected from `BilevelConfig().reporter(config=...)`.

The hooks push benchmark fields with `self._log(key, value)`, one value per
step, under the exact field names of the schema; compare
`examples/bilevel_fishery/regulated_env.py` with
`examples/bilevel_fishery/metric_schema.py`. In a bilevel configuration the
schema and the queries are declared once, on the inner optimizer:
`APPOptimizerConfig().environment(env=..., env_config=..., schema=..., queries=...)`,
exactly as `debug.py::build_config` does with `FisheryMetricSchema` (section 24
shows it on this benchmark). Below, a two-field schema, three queries, and a
subclass of the benchmark whose transition hook logs into the schema. A
subclass may declare its own hook of a type the parent already declares: the
discovery in `__init_subclass__` looks at the subclass body only, so the new
hook replaces the inherited one.


In [ ]:
from typing import Optional

from pydantic import Field

from core.envs.schema import EpisodeRolloutSchema
from core.metrics.enums import ReduceProtocol
from core.reporting.query import Query


class RenewableMetricSchema(EpisodeRolloutSchema):
    """Episode schema of the renewable benchmark: stock and realized harvest per step."""

    resource_norm: Optional[float] = Field(
        default=None, json_schema_extra={"reduce": ReduceProtocol.MEAN}
    )
    realized_harvest: Optional[float] = Field(
        default=None, json_schema_extra={"reduce": ReduceProtocol.MEAN}
    )


RENEWABLE_QUERIES = (
    Query(title="Resource level", x=("iter",), y=("resource_norm",)),
    Query(title="Realized harvest", x=("iter",), y=("realized_harvest",)),
    Query(title="Reward — all agents", x=("iter",), y=("by_agent", "*", "reward")),
)


class LoggedRenewableResourceEnv(RenewableResourceEnv):
    """Same dynamics; the transition hook also pushes two schema fields per step."""

    @transition
    def logged_transition(
        self, *, A_t: MultiAgentDict, S_t: dict[str, float], **kwargs
    ) -> dict[str, float]:
        new_state = self.renewable_transition(A_t=A_t, S_t=S_t, **kwargs)
        self._log(("resource_norm",), float(S_t["resource"]) / self.K)
        self._log(("realized_harvest",), new_state["last_usage"])
        return new_state


print("transition hook in force:", LoggedRenewableResourceEnv._transition)


In [ ]:
# A published candidate is handed to one environment only: publish another copy.
ray.get(
    world.append_context.remote(
        Context(
            id=None,
            opt_id="tutorial_outer",
            step=0,
            env="tutorial",
            payload=MechanismContext(
                index=1,
                env_id=None,
                seed=POLICY_SEED,
                status=MechanismStatus.published,
                mechanism=mechanism,
                metrics=None,
            ),
        )
    )
)

logged_env = LoggedRenewableResourceEnv(
    world=world,
    opt_id="tutorial_logged",
    mechanism_id=1,
    agents=AGENTS,
    mechanism=mechanism,
    horizon=5,
    seed=7,
    policy_seed=POLICY_SEED,
    ecology_cfg=FIXTURE,
    action_spaces={a: spaces.Box(-np.inf, np.inf, (1,), np.float32) for a in AGENTS},
    observation_spaces={
        a: spaces.Box(0.0, 1.0, (OBS_DIM,), np.float32) for a in AGENTS
    },
    schema=RenewableMetricSchema,  # -> env.logger is a MetricLogger
    queries=RENEWABLE_QUERIES,  # rendered only when a reporter_cfg is also given
)
obs, _ = logged_env.reset()
assert logged_env.published_mechanism_assigned

policy_rng = np.random.default_rng(0)
for _ in range(logged_env.horizon):
    logged_env.step({a: policy_rng.normal(size=1).astype(np.float32) for a in AGENTS})

metrics = logged_env.logger.peek()  # non-destructive: raw per-step histories
print(type(metrics).__name__)
print("iter            :", metrics.iter)
print("resource_norm   :", np.round(metrics.resource_norm, 4))
print("realized_harvest:", np.round(metrics.realized_harvest, 3))
print("reward_mean     :", np.round(metrics.reward_mean, 4))
print(
    "by_agent reward :",
    {a: np.round(s.reward, 4).tolist() for a, s in metrics.by_agent.items()},
)

assert len(metrics.resource_norm) == logged_env.horizon == len(metrics.iter)
assert np.isclose(metrics.resource_norm[0], FIXTURE["x_init"] / FIXTURE["K"])
assert sorted(metrics.by_agent) == AGENTS


# 24. Plugging the benchmark into a bilevel configuration

`BilevelConfig().mechanism(mechanism=<Mechanism>)` sets the mechanism
template shared by both levels: its `dimension`/`encode`/`decode` define the
outer optimizer's search space, and it is the default mechanism of the
regulated environments until a candidate is published. The inner level
receives the benchmark class and its `env_config` through the RLlib-side
configuration, exactly as `examples/bilevel_fishery/debug.py::build_config`
does for `FisheryRegulatedEnv`:

```python
BilevelConfig()
    .world(world_name="renewable_world")
    .mechanism(mechanism=mechanism)
    .training(outer_iters=100)
    .ray(device="cpu", num_cpus=4, omp_threads=1)
    .outer(ESConfig().training(sigma=0.15, mean_lr=0.10)
                     .environment(env=<your RegulatorEnv>, env_config={...},
                                  horizon=100, train_iters=50))
    .inner(APPOptimizerConfig().environment(env=RenewableResourceEnv,
                                            env_config={"ecology_cfg": FIXTURE, ...}))
```

The outer level also needs a regulator environment subclassing
`core.envs.regulator.RegulatorEnv` that aggregates the published step contexts
into a fitness. This notebook does not build one:
`examples/bilevel_fishery/regulator_env.py::FisheryRegulatorEnv` is the
reference implementation, `examples/bilevel_fishery/debug.py::build_config`
shows the complete outer and inner wiring around it, and section 11 of
`mechanism_algorithms.ipynb` walks through that configuration. Below we only
build the configuration object, without `build_optimizer()`, to show the
template wiring and, on the inner level, the metric schema and queries of
section 23.

In [ ]:
from core.optimizers.appo.config import APPOptimizerConfig
from core.optimizers.bilevel import BilevelConfig

bilevel_cfg = (
    BilevelConfig()
    .world(world_name="renewable_world")
    .mechanism(mechanism=mechanism)
    .training(outer_iters=3)
    .inner(
        APPOptimizerConfig().environment(
            env=LoggedRenewableResourceEnv,
            env_config={"ecology_cfg": FIXTURE},
            horizon=5,
            schema=RenewableMetricSchema,
            queries=RENEWABLE_QUERIES,
        )
    )
)

assert bilevel_cfg.mechanism_template is mechanism
print("outer search space dimension:", bilevel_cfg.mechanism_template.dimension)
print("parameters:", bilevel_cfg.mechanism_template.param_names())
print(
    "inner env schema / queries :",
    bilevel_cfg.inner_cfg._reporting_schema_env.__name__,
    [q.title for q in bilevel_cfg.inner_cfg._reporting_queries_env],
)

In [ ]:
ray.shutdown()

# 25. Reproducibility checklist

Record:

```text
benchmark constants
mechanism parameters
action-component semantics
env seed
policy seed
evaluation seeds
horizon
number of agents
stochastic-noise model
optimizer seed
```

Validate in this order:

1. deterministic transition (section 15);
2. deterministic mechanism transform (section 16);
3. deterministic full environment step (section 18);
4. repeated seeded episode (section 21);
5. inner RL;
6. outer optimization.

# 26. Minimal test suite for a new benchmark

```text
test_reset_state
test_transition_matches_equation
test_transition_boundaries
test_base_reward
test_observation_shape
test_action_semantics
test_info_values
test_context_publication

test_mechanism_required_bindings
test_mechanism_transform
test_mechanism_encode_decode
test_mechanism_clip
test_mechanism_param_names

test_benchmark_plus_mechanism_one_step
test_benchmark_plus_mechanism_episode
```

These names are a template for the reader's own test module: none of them
exists in `tests/` yet. Sections 15 to 23 of this notebook are their
prototypes. Unit tests can avoid Ray entirely by calling the hooks and the
mechanism channels directly (sections 15 and 16); integration tests use a real
`World` actor as in `tests/integration/test_fishery_mechanisms.py`.

The fishery benchmark's real tests follow the same layout under other names:
`tests/examples/test_fishery_regulated_env.py` (deterministic reset,
closed-form transition, stock bounds, quota cap, subsidy, every logged field
the queries reference), `tests/mechanism/algorithms/test_quota.py` and its
siblings (required bindings, ranges, encode/decode round trip, clip, channel
transforms), `tests/mechanism/composition/test_chained_mechanism.py`,
`tests/envs/test_marl_regulated.py` (action pipeline, inert step, observation
dimension) and, with a `World` actor,
`tests/integration/test_fishery_mechanisms.py::test_stage_a_environment_only`
and `::test_stage_c_bilevel_smoke`, plus
`tests/integration/test_fishery_reporting.py` for the CSV output of a run.

Only after these pass should the benchmark be used in an expensive bilevel
run.

# 27. Pull-request checklist for a new benchmark

## Theory

- state is mathematically defined;
- action semantics are mathematically defined;
- transition equation is documented;
- base reward is documented;
- observation is documented;
- mechanism intervention point is documented.

## Code

- reset hook;
- transition hook;
- reward hook;
- observation hook;
- action/observation spaces (including mechanism augmentation);
- info/context fields;
- mechanism bindings;
- optimizer-vector semantics for learned mechanism parameters.

## Tests

- equations tested numerically;
- mechanism tested independently;
- composition tested;
- deterministic end-to-end step tested;
- seeded reproducibility tested.

This order keeps the code tied to a scientific model rather than letting the
class hierarchy define the experiment.

# 28. Summary

A new benchmark is a subclass of `MultiAgentRegulatedEnv` that owns the
equations of its world through four hooks (reset, transition, base reward,
observation) and nothing about regulation. A new mechanism is a frozen
dataclass that satisfies the optimizer contract (`dimension`, `encode`,
`decode`, `clip`, `param_names`, `to_vector`) and overrides only the channels it
uses; whatever it needs from the world arrives through bindings, which is why
the existing `QuotaMechanism` plugged into the renewable resource without a
line of fishery code. Composition concatenates optimizer vectors and applies
channels in order, and the regulated step applies the action channel before
the transition and the reward channel after it. Every equation was tested
before any policy was trained: the transition as a pure function, the tax
analytically, encode/decode round trips, then one complete step against a real
`World` actor and a seeded episode. Metrics are opt-in: with a `schema` the
environment logs one value per step through `self._log`, and the same schema
and queries are declared on the inner optimizer of a bilevel configuration.
The outer level needs a regulator environment, documented with the fishery
example rather than here.
